### 해양수산부 국립해양조사원_연안 침수 정보 조회

이 데이터는 해양수산부 국립해양조사원에서 2009년~2020년까지 제작된 100년빈도 규모의 연안 인접지역 침수범위 예측정보를 제공하기 위해 수집한 데이터입니다.
검색 대상 시군구코드 등 값을 파라미터로 데이터를 조회할 수 있으며, 본 데이터의 주요 내용은 시도, 시군구명과 예측 침수값 및 침수시 공간정보 등 정보로 구성되어 있습니다.
본 데이터는 지방자치단체에서 침수 취약 지역의 방재 대책 수립 시 활용할 수 있으며, 재난관리 담당 부서가 안전 진단 및 예방 활동의 기초자료로 사용할 수 있습니다. 또한 침수 위험지구 관리와 연안 개발계획 수립 등 정책 수립 과정에서도 사용할 수 있습니다.

* 분류체계: 해양수산 - 해양수산·어촌
* 제공기관: 해양수산부 국립해양조사원
* 관리부서명:    해양예보과	관리부서 전화번호	051-400-4387
* API 유형:   REST	데이터포맷	JSON+XML
* End Point: https://apis.data.go.kr/1192136/waterlogged
* Service:  /GetWaterloggedApiService - 연안침수정보:우리나라 연안에 대한 조위별 침수 정보 제공 서비스

In [1]:
%useLatestDescriptors

%use dataframe
%use kandy

##### 시군구 코드 정보
* 출처 : https://www.data.go.kr/cmm/cmm/fileDownload.do?atchFileId=FILE_000000003661213&fileDetailSn=1
* <오픈API 활용가이드_연안 침수 정보> 내용중 [ 연안 침수 정보 대상 시군구 코드 ] 사용

In [2]:
val url_code = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/KotlinNotebooks/data/sggCode.csv"
val df_code = DataFrame.readCsv(url_code, ',')

kotlin-logging: initializing... active logger factory: Slf4jLoggerFactory


In [3]:
val codeList = df_code.시군구코드.toList()

In [4]:
val serviceKeyFilePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/collectionServer/src/main/resources/application.json"
val configData = DataRow.readJson(path=serviceKeyFilePath)
val config = configData["WATER_LOGGED"]

In [5]:
val numOfRows = 300

val allDataFrames = mutableListOf<DataFrame<*>>()

codeList.forEachIndexed{ index, it ->
    val baseUrl = "${config.endPoint}/${config.subPath}" +
            "?serviceKey=${config.apikey}&type=json&sggCd=${it}&numOfRows=${numOfRows}"

    val firstUrl = "${baseUrl}&pageNo=1"

    val df_first = DataFrame.readJson(firstUrl)
    val data = df_first["body"]["items"]["item"].first() as DataFrame<*>

    val totalCount = (df_first["body"]["totalCount"][0] as Number).toInt()
    val totalPages = ceil(totalCount.toDouble() / numOfRows).toInt()
 //   println("ssgNm:${it}, 시군구:${data[0][0]}/${data[0][1]}, 총 데이터 개수: $totalCount, 전체 페이지 수: $totalPages")

    val dataFrames = mutableListOf<DataFrame<*>>()

    dataFrames.add(data)

    for (page in 2..totalPages) {
        val url = "$baseUrl&pageNo=$page"
        val df_page = DataFrame.readJson(url)
        val data = df_page["body"]["items"]["item"].first() as DataFrame<*>
        dataFrames.add(data)
    }

    val df_SggAll = dataFrames.concat()

    allDataFrames.add(df_SggAll)

}

val df = allDataFrames.concat()

In [6]:
val countsDf = df.groupBy { flodVlCn }
    .count()
    .sortByDesc("flodVlCn")


In [7]:
val joinedDf = df.join(df_code){ (ctpvNm match right.시도명) and  (sggNm match right.시군구명) }
    .add("alertLevel"){
        when(flodVlCn.trim()){
            "0.0-0.5" -> 1
            "0.5-1.0" -> 2
            "1.0-1.5" -> 3
            "1.5-2.0" -> 4
            "2.0-2.5" -> 7
            "2.5-3.0" -> 7
            "2.0-3.0" -> 7
            "3.0" -> 8
            else -> 0
        }
    }.rename("시군구코드").to("areaCode")

* WKT(Well-Known Text) 형식의 MULTIPOLYGON 문자열 리스트를을 GeoJSON 형식으로 변환하는 Kotlin 코드

In [8]:
// 1. 각 WKT에서 "MULTIPOLYGON" 키워드를 제거하고 가장 겉의 괄호 1쌍을 제거
val extractedPolygons = joinedDf.geom.map { wkt ->
    wkt.trim()
        .replace(Regex("""^MULTIPOLYGON\s*""", RegexOption.IGNORE_CASE), "")
        .removePrefix("(")
        .removeSuffix(")")
        .trim()
}


In [13]:
fun parseWktToLatLngList(wktString: String): List<Map<String,Double>> {
    // 1. 괄호 제거 및 공백 정리
    val cleaned = wktString.replace("(", "").replace(")", "").trim()

    // 2. 쉼표(,)를 기준으로 각 좌표 쌍 분리
    val coordinatePairs = cleaned.split(",")

    // 3. 각 쌍을 공백으로 분리하여 LatLng 객체로 변환
    return coordinatePairs.mapNotNull { pair ->
        val parts = pair.trim().split("\\s+".toRegex())
        if (parts.size == 2) {
            val lng = parts[0].toDoubleOrNull()
            val lat = parts[1].toDoubleOrNull()

            if (lat != null && lng != null) {
                mapOf("lat" to lat, "lng" to lng) // 구글 맵 포맷인 (위도, 경도) 순서로 생성
              //  "{lat:${lat}, lng:${lng}}"
            } else null
        } else null
    }
}

In [10]:
val latlngMapList = extractedPolygons.map { poligonString ->
    parseWktToLatLngList(poligonString)
}